In [1]:
from constants import DATA_ROOT_PATH_NAME, BANDPASS, HAMPEL_WINDOW_SIZE, HAMPEL_N_SIGMA, CROP_TMIN, CROP_TMAX, LOCAL_DETREND_WINDOW_SEC, LOCAL_DETREND_STEP_SEC, ASR_CUTOFF, ASR_BLOCKSIZE, ASR_WIN_LEN, ASR_WIN_OVERLAP, ASR_MAX_DROPOUT_FRACTION, ASR_MIN_CLEAN_FRACTION, ASR_MAX_BAD_CHANS

from preprocessing.step.bandpass import BandpassFilterStep
from preprocessing.step.detrend import LocalDetrendStep
from preprocessing.step.hampel import HampelFilterStep
from preprocessing.step.asr import ASRStep
from preprocessing.step.crop import CropStep

from preprocessing.pipeline import PreprocessingPipeline
import numpy as np

from features.factory import FeatureExtractionEngine, FeatureExtractionConfig
from features.categories import FeatureCategory

from eeg.data import EEGRecordedDataProvider

from features.visualization import TopomapFactory



%load_ext autoreload
%autoreload 2

# Création des 2 datasets de features

In [ ]:
recordings = EEGRecordedDataProvider.build(DATA_ROOT_PATH_NAME)

/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-eyesclosed.

The search_str was "data/sub-001/**/eeg/sub-001*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 57
Group: A
MMSE: 16
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-002_task-eyesclosed.

The search_str was "data/sub-002/**/eeg/sub-002*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 78
Group: A
MMSE: 22
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not fin

In [ ]:
asr_pipeline = PreprocessingPipeline(name="ASR",
                                        steps=[
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                ASRStep(cutoff=ASR_CUTOFF, blocksize=ASR_BLOCKSIZE, win_len=ASR_WIN_LEN, win_overlap=ASR_WIN_OVERLAP, max_dropout_fraction=ASR_MAX_DROPOUT_FRACTION, min_clean_fraction=ASR_MIN_CLEAN_FRACTION, max_bad_chans=ASR_MAX_BAD_CHANS)
                                                ])

dethamp_pipeline = PreprocessingPipeline(name="det-hamp",
                                        steps=[ 
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                LocalDetrendStep(window_sec=LOCAL_DETREND_WINDOW_SEC, step_sec=LOCAL_DETREND_STEP_SEC),
                                                HampelFilterStep(window_size=HAMPEL_WINDOW_SIZE, n_sigma=HAMPEL_N_SIGMA)
                                                ])

In [4]:
recorded_eeg = recordings[14]
asr_processed_eeg = asr_pipeline.compute(recorded_eeg)
dethamp_processed_eeg = dethamp_pipeline.compute(recorded_eeg)

In [ ]:
categories_to_extract = [FeatureCategory.WAVELET, FeatureCategory.TEMPORAL, FeatureCategory.POWER_RATIO, FeatureCategory.SPECTRAL]
config = FeatureExtractionConfig(categories_to_extract=categories_to_extract, wamp_threshold=10e-9, ppc_epoch_duration=2)
feature_extraction_engine = FeatureExtractionEngine(config=config)

dethamp_processed_extraction_result = feature_extraction_engine.extract(dethamp_processed_eeg)


asr_processed_extraction_result = feature_extraction_engine.extract(asr_processed_eeg)


/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(
/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(


# Import de toutes les features de tous les participants

In [2]:
from features.io import FeaturesDatasetIO
dethamp_dataset = FeaturesDatasetIO.load("computed_data/dethamp")
asr_dataset = FeaturesDatasetIO.load("computed_data/asr")

In [3]:
dethamp_dataset.participant_datasets[0].ppc_raw_data

array([[[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ]],

       [[0.49348072, 0.88798414, 0.97666357, 0.9652053 , 0.93349314,
         0.91102336],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.   

# Tests statistiques

In [3]:
from stats.queries import QueryFactoryConfig, QueryFactory
from stats.runner import StatisticalTestRunner
from stats.correction.fdr import FDRCorrector

factory = QueryFactory(
    QueryFactoryConfig.from_lists(
        subject_variables={"subject_age", "subject_mmse", "subject_id", "subject_mmse", "subject_health"},
        eeg_features=dethamp_dataset.feature_names,
    )
)

tttest_queries = { 
    "subject_age":
    factory.compare_groups(
        target="subject_age",
        group_col="subject_health",
        group_a="Healthy",
        group_b="Alzheimer",
        test_kind="t_test",
        scope="subject",
    ),

    "subject_mmse" :
    factory.compare_groups(
        target="subject_mmse",
        group_col="subject_health",
        group_a="Healthy",
        group_b="Alzheimer",
        test_kind="t_test",
        scope="subject",
    )
}

features_group_comparaison_queries = {
    feature_name : 
    factory.compare_groups(
        target=feature_name,
        group_col="subject_health",
        group_a="Healthy",
        group_b="Alzheimer",
        test_kind="wilcoxon_rank_sum",
        scope="all_channels"
    )
for feature_name in dethamp_dataset.feature_names}


correlation_queries = {
    feature_name : 
    factory.correlate(
        x=feature_name,
        y="subject_mmse",
        test_kind="spearman",
        scope="all_channels",
    )
    for feature_name in dethamp_dataset.feature_names
}

raw_result_set = StatisticalTestRunner.run(features_group_comparaison_queries, dethamp_dataset)
corrected_result_set = FDRCorrector.correct(raw_result_set)

corrected_result_set

{'theta_beta_ratio': CorrectedStatisticalTestResultSet(results={'Fp1': CorrectedStatisticalTestResult(statistic=-2.7185393664628092, p_value=0.006557085256156329, target='theta_beta_ratio', key='Fp1', test_name='wilcoxon-rank-sum', n_x=29, n_y=36, x_name='theta_beta_ratio (Healthy, Fp1)', y_name='theta_beta_ratio (Alzheimer, Fp1)', p_value_corrected=0.007786538741685641, correction_method='fdr_bh', alpha=0.05, reject_null=True), 'Fp2': CorrectedStatisticalTestResult(statistic=-2.5337842638876666, p_value=0.01128381774175837, target='theta_beta_ratio', key='Fp2', test_name='wilcoxon-rank-sum', n_x=29, n_y=36, x_name='theta_beta_ratio (Healthy, Fp2)', y_name='theta_beta_ratio (Alzheimer, Fp2)', p_value_corrected=0.01191069650518939, correction_method='fdr_bh', alpha=0.05, reject_null=True), 'F3': CorrectedStatisticalTestResult(statistic=-3.7610860167082554, p_value=0.00016917723592868613, target='theta_beta_ratio', key='F3', test_name='wilcoxon-rank-sum', n_x=29, n_y=36, x_name='theta_be

In [ ]:
print("hello")

hello
